# Kaggle RAG Training Lab

Notebook ini memproses `corpus_export.jsonl` dari lokal menjadi:

- `cleaned_chunks.parquet`
- `embeddings.npy`
- `metadata.jsonl`
- `retrieval_score.csv`
- `evaluation_report.md`

Upload `corpus_export.jsonl` ke Kaggle Notebook sebagai input dataset.


In [ ]:
!pip -q install sentence-transformers pandas pyarrow numpy tqdm


In [ ]:
from pathlib import Path
import json
import re
import hashlib

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer


## 1. Cari file corpus_export.jsonl


In [ ]:
# Kaggle biasanya menaruh dataset input di /kaggle/input/<nama-dataset>/
candidates = list(Path("/kaggle/input").rglob("corpus_export.jsonl"))
if not candidates:
    raise FileNotFoundError("corpus_export.jsonl tidak ditemukan. Upload dulu sebagai Kaggle Dataset atau input notebook.")

CORPUS_PATH = candidates[0]
OUT_DIR = Path("/kaggle/working")
print("Corpus:", CORPUS_PATH)


In [ ]:
def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

corpus = read_jsonl(CORPUS_PATH)
len(corpus), corpus[0].keys()


## 2. Cleaning sederhana


In [ ]:
def clean_text(text: str) -> str:
    text = text.replace("\x00", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

for row in corpus:
    row["text"] = clean_text(row["text"])

print("Jumlah dokumen:", len(corpus))


## 3. Chunking berbasis paragraf

Parameter awal mengikuti rencana project: sekitar 700 token/karakter fleksibel dengan overlap.
Di sini digunakan pendekatan karakter agar ringan dan stabil.


In [ ]:
CHUNK_SIZE = 1800       # kira-kira mendekati 600-800 token untuk bahasa Indonesia, tergantung teks
CHUNK_OVERLAP = 300
MIN_CHUNK_CHARS = 120

def paragraph_chunks(text: str, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    chunks = []
    current = ""

    for p in paragraphs:
        candidate = (current + "\n\n" + p).strip() if current else p
        if len(candidate) <= chunk_size:
            current = candidate
        else:
            if len(current) >= MIN_CHUNK_CHARS:
                chunks.append(current)
            # fallback jika paragraf sangat panjang
            if len(p) > chunk_size:
                start = 0
                while start < len(p):
                    end = min(start + chunk_size, len(p))
                    piece = p[start:end].strip()
                    if len(piece) >= MIN_CHUNK_CHARS:
                        chunks.append(piece)
                    if end >= len(p):
                        break
                    start = max(0, end - overlap)
                current = ""
            else:
                current = p

    if len(current) >= MIN_CHUNK_CHARS:
        chunks.append(current)

    return chunks

chunk_rows = []
metadata_rows = []

for row in corpus:
    chunks = paragraph_chunks(row["text"])
    for i, chunk in enumerate(chunks):
        base_id = f"{row['doc_id']}_{i:04d}"
        chunk_id = hashlib.sha1(base_id.encode("utf-8")).hexdigest()[:24]
        meta = {
            "chunk_id": chunk_id,
            "doc_id": row.get("doc_id", ""),
            "title": row.get("title", ""),
            "source": row.get("source", ""),
            "chunk_index": i,
            "origin": "kaggle_training_lab",
        }
        chunk_rows.append({
            "chunk_id": chunk_id,
            "doc_id": row.get("doc_id", ""),
            "title": row.get("title", ""),
            "source": row.get("source", ""),
            "chunk_index": i,
            "text": chunk,
        })
        metadata_rows.append(meta)

chunks_df = pd.DataFrame(chunk_rows)
chunks_df.head(), len(chunks_df)


## 4. Embedding batch

Model default: `intfloat/multilingual-e5-small`.

Penting: di lokal harus memakai model yang sama untuk query embedding.


In [ ]:
MODEL_NAME = "intfloat/multilingual-e5-small"
PASSAGE_PREFIX = "passage:"

model = SentenceTransformer(MODEL_NAME)
texts = [f"{PASSAGE_PREFIX} {t}" for t in chunks_df["text"].astype(str).tolist()]

embeddings = model.encode(
    texts,
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

embeddings.shape


## 5. Simpan output utama


In [ ]:
chunks_path = OUT_DIR / "cleaned_chunks.parquet"
emb_path = OUT_DIR / "embeddings.npy"
meta_path = OUT_DIR / "metadata.jsonl"

chunks_df.to_parquet(chunks_path, index=False)
np.save(emb_path, embeddings)

with open(meta_path, "w", encoding="utf-8") as f:
    for row in metadata_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(chunks_path)
print(emb_path)
print(meta_path)


## 6. Evaluasi retrieval sederhana

Buat query dummy dulu. Nanti bisa diganti dengan 50-100 pertanyaan evaluasi.


In [ ]:
QUERY_PREFIX = "query:"

eval_queries = [
    "apa isi utama dokumen ini?",
    "sebutkan poin penting dari corpus",
]

query_emb = model.encode(
    [f"{QUERY_PREFIX} {q}" for q in eval_queries],
    normalize_embeddings=True
).astype("float32")

# cosine similarity karena embedding sudah normalized = dot product
scores = query_emb @ embeddings.T
top_k = 5

eval_rows = []
for qi, q in enumerate(eval_queries):
    top_idx = np.argsort(-scores[qi])[:top_k]
    for rank, idx in enumerate(top_idx, start=1):
        eval_rows.append({
            "query": q,
            "rank": rank,
            "score": float(scores[qi, idx]),
            "chunk_id": chunks_df.iloc[idx]["chunk_id"],
            "title": chunks_df.iloc[idx]["title"],
            "source": chunks_df.iloc[idx]["source"],
            "chunk_preview": chunks_df.iloc[idx]["text"][:250],
        })

retrieval_df = pd.DataFrame(eval_rows)
retrieval_df.to_csv(OUT_DIR / "retrieval_score.csv", index=False)
retrieval_df


In [ ]:
report = f"""# Evaluation Report

## Corpus
- Documents: {len(corpus)}
- Chunks: {len(chunks_df)}
- Embedding model: {MODEL_NAME}
- Embedding shape: {embeddings.shape}

## Chunking
- CHUNK_SIZE: {CHUNK_SIZE}
- CHUNK_OVERLAP: {CHUNK_OVERLAP}
- MIN_CHUNK_CHARS: {MIN_CHUNK_CHARS}

## Output
- cleaned_chunks.parquet
- embeddings.npy
- metadata.jsonl
- retrieval_score.csv

## Catatan
Evaluasi ini masih dummy. Untuk evaluasi serius, buat 50-100 pertanyaan dengan expected source/chunk lalu hitung hit@3 atau hit@5.
"""

(OUT_DIR / "evaluation_report.md").write_text(report, encoding="utf-8")
print(report)
